# Regression Failure Classifier — Data Analysis

Exploratory analysis of `TC_MASTER` and `HIS_EXEC_REPORT` tables to understand the data before building the classifier.

In [28]:
import oracledb
import os
from dotenv import load_dotenv

load_dotenv()

connection = oracledb.connect(
    user     = os.getenv('DB_USER'),
    password = os.getenv('DB_PASSWORD'),
    dsn      = f"{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_SERVICE')}"
)

print("Connected to Oracle DB ✓")
print(f"  Host    : {os.getenv('DB_HOST')}")
print(f"  Service : {os.getenv('DB_SERVICE')}")
print(f"  User    : {os.getenv('DB_USER')}")

Connected to Oracle DB ✓
  Host    : ny-oracle-ee11
  Service : AUTOBOT_DEV
  User    : autobot_data


In [29]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 120)

tc_master = pd.read_csv('TC_MASTER.csv')
his_exec  = pd.read_csv('HIS_EXEC_REPORT.csv')

print('TC_MASTER shape     :', tc_master.shape)
print('HIS_EXEC_REPORT shape:', his_exec.shape)

TC_MASTER shape     : (33890, 14)
HIS_EXEC_REPORT shape: (13624, 16)


---
## 1. TC_MASTER — Overview

In [30]:
tc_master.head(5)

,TC_ID,JIRA_ID,MODULE,J_COMPONENT,BASELINE_TIME,J_LABEL,FIXED_VERSION,CONTINENT,AUTOMATED_TC_ID,AUTOMATED_BY_USERID,FUNC_AREA,MAINTENANCE_DONE_BY,DEPENDENCY,SUB_COMPONENT
0,75,IAPP-291777,XSYSTEMS,OP - X-Systems - Client maintenance - Billing References,0,"[""1P-API-AUTOMATION"",""1P-XSYSTEM"",""ONA"",""smoke-test""]",2021.2,US,billingReferences.IAPP_291781_291779_291777,Aparna Misal,Maintenance,NaN,NaN,NaN
1,76,IAPP-291779,XSYSTEMS,OP - X-Systems - Client maintenance - Billing References,0,"[""1P-API-AUTOMATION"",""1P-XSYSTEM"",""ONA"",""smoke-test""]",2021.2,US,billingReferences.IAPP_291781_291779_291777,Aparna Misal,Maintenance,NaN,NaN,NaN
2,77,IAPP-291781,XSYSTEMS,OP - X-Systems - Client maintenance - Billing References,0,"[""1P-API-AUTOMATION"",""1P-XSYSTEM"",""ONA"",""smoke-test""]",2021.2,US,billingReferences.IAPP_291781_291779_291777,Aparna Misal,Maintenance,NaN,NaN,NaN
3,128,IAPP-298055,XSYSTEMS,OP - X-Systems - Vendor/Remit maintenance,0,"[""1P-XSYSTEM"",""ONA""]",2021.3,US,vendorRemit.IAPP_298055_298057_298323_298066_298068,Poonam Bhagat,Maintenance,NaN,NaN,NaN
4,129,IAPP-298057,XSYSTEMS,OP - X-Systems - Vendor/Remit maintenance,0,"[""1P-XSYSTEM"",""ONA""]",2021.3,US,vendorRemit.IAPP_298055_298057_298323_298066_298068,Poonam Bhagat,Maintenance,NaN,NaN,NaN


In [21]:
# Column fill rates — TC_MASTER
total_rows_tc = len(tc_master)
filled_tc     = (tc_master.notna() & (tc_master.fillna('') != '')).sum()
missing_tc    = total_rows_tc - filled_tc
fill_pct_tc   = (filled_tc / total_rows_tc * 100).round(1)

fill_summary_tc = pd.DataFrame({
    'Column'     : filled_tc.index,
    'Total Rows' : total_rows_tc,
    'Filled'     : filled_tc.values,
    'Missing'    : missing_tc.values,
    'Fill %'     : fill_pct_tc.values
}).sort_values('Fill %', ascending=False).reset_index(drop=True)

fill_summary_tc

,Column,Total Rows,Filled,Missing,Fill %
0,TC_ID,33890,33890,0,100.0
1,JIRA_ID,33890,33890,0,100.0
2,MODULE,33890,33890,0,100.0
3,BASELINE_TIME,33890,33890,0,100.0
4,J_LABEL,33890,33886,4,100.0
5,CONTINENT,33890,33890,0,100.0
6,AUTOMATED_TC_ID,33890,33884,6,100.0
7,FIXED_VERSION,33890,33796,94,99.7
8,J_COMPONENT,33890,32417,1473,95.7
9,AUTOMATED_BY_USERID,33890,31961,1929,94.3


In [31]:
# Column fill rates — TC_MASTER
total_rows_tc = len(tc_master)
filled_tc     = (tc_master.notna() & (tc_master.fillna('') != '')).sum()
missing_tc    = total_rows_tc - filled_tc
fill_pct_tc   = (filled_tc / total_rows_tc * 100).round(1)

fill_summary_tc = pd.DataFrame({
    'Column'     : filled_tc.index,
    'Total Rows' : total_rows_tc,
    'Filled'     : filled_tc.values,
    'Missing'    : missing_tc.values,
    'Fill %'     : fill_pct_tc.values
}).sort_values('Fill %', ascending=False).reset_index(drop=True)

fill_summary_tc

,Column,Total Rows,Filled,Missing,Fill %
0,TC_ID,33890,33890,0,100.0
1,JIRA_ID,33890,33890,0,100.0
2,MODULE,33890,33890,0,100.0
3,BASELINE_TIME,33890,33890,0,100.0
4,J_LABEL,33890,33886,4,100.0
5,CONTINENT,33890,33890,0,100.0
6,AUTOMATED_TC_ID,33890,33884,6,100.0
7,FIXED_VERSION,33890,33796,94,99.7
8,J_COMPONENT,33890,32417,1473,95.7
9,AUTOMATED_BY_USERID,33890,31961,1929,94.3


---
## 2. HIS_EXEC_REPORT — Overview

In [12]:
his_exec.head(5)

,EXEC_ID,JIRA_ID,INTRIM_STATUS,TC_ID,ASSIGNEE,USER_REMARKS,AUTO_FAILURE_REASON,FINAL_STATUS,MODULE,BUILD_NO,FAILURE_REMARKS,DEFECTID,J_COMPONENT,EXEC_TIME,START_TIME,END_TIME
0,UI_NTV,IAPP-373498,MAINTAINED,22405,autobot,DB: Formal run trial but proceed with Oneview later,ENV-ISSUE,PASSED,Prisma,NaN,NaN,NaN,Orders,NaN,NaN,NaN
1,UI_NTV,IAPP-372969,MAINTAINED,22408,autobot,DB: Formal run trial but proceed with Oneview later,ENV-ISSUE,PASSED,Prisma,NaN,NaN,NaN,Orders,NaN,NaN,NaN
2,UI_NTV,IAPP-370479,MAINTAINED,22411,autobot,DB: Formal run trial but proceed with Oneview later,ENV-ISSUE,PASSED,Prisma,NaN,NaN,NaN,Orders,NaN,NaN,NaN
3,UI_NTV,IAPP-370477,MAINTAINED,22412,autobot,DB: Formal run trial but proceed with Oneview later,ENV-ISSUE,PASSED,Prisma,NaN,NaN,NaN,Orders,NaN,NaN,NaN
4,UI_NTV,IAPP-370304,MAINTAINED,22418,autobot,DB: Formal run trial but proceed with Oneview later,ENV-ISSUE,PASSED,Prisma,NaN,NaN,NaN,Orders,NaN,NaN,NaN


In [13]:
# Column fill rates — HIS_EXEC_REPORT
total_rows = len(his_exec)
filled     = (his_exec.notna() & (his_exec.fillna('') != '')).sum()
missing    = total_rows - filled
fill_pct   = (filled / total_rows * 100).round(1)

fill_summary = pd.DataFrame({
    'Column'     : filled.index,
    'Total Rows' : total_rows,
    'Filled'     : filled.values,
    'Missing'    : missing.values,
    'Fill %'     : fill_pct.values
}).sort_values('Fill %', ascending=False).reset_index(drop=True)

fill_summary

,Column,Total Rows,Filled,Missing,Fill %
0,EXEC_ID,13624,13624,0,100.0
1,JIRA_ID,13624,13624,0,100.0
2,INTRIM_STATUS,13624,13624,0,100.0
3,TC_ID,13624,13624,0,100.0
4,ASSIGNEE,13624,13624,0,100.0
5,FINAL_STATUS,13624,13624,0,100.0
6,J_COMPONENT,13624,13624,0,100.0
7,MODULE,13624,13624,0,100.0
8,USER_REMARKS,13624,6128,7496,45.0
9,AUTO_FAILURE_REASON,13624,6104,7520,44.8


In [32]:
# Column fill rates — HIS_EXEC_REPORT
total_rows = len(his_exec)
filled     = (his_exec.notna() & (his_exec.fillna('') != '')).sum()
missing    = total_rows - filled
fill_pct   = (filled / total_rows * 100).round(1)

fill_summary = pd.DataFrame({
    'Column'     : filled.index,
    'Total Rows' : total_rows,
    'Filled'     : filled.values,
    'Missing'    : missing.values,
    'Fill %'     : fill_pct.values
}).sort_values('Fill %', ascending=False).reset_index(drop=True)

fill_summary

,Column,Total Rows,Filled,Missing,Fill %
0,EXEC_ID,13624,13624,0,100.0
1,JIRA_ID,13624,13624,0,100.0
2,INTRIM_STATUS,13624,13624,0,100.0
3,TC_ID,13624,13624,0,100.0
4,ASSIGNEE,13624,13624,0,100.0
5,FINAL_STATUS,13624,13624,0,100.0
6,J_COMPONENT,13624,13624,0,100.0
7,MODULE,13624,13624,0,100.0
8,USER_REMARKS,13624,6128,7496,45.0
9,AUTO_FAILURE_REASON,13624,6104,7520,44.8


In [25]:
# AUTO_FAILURE_REASON — Classification status
total        = len(his_exec)
classified   = his_exec['AUTO_FAILURE_REASON'].fillna('').str.strip().ne('').sum()
unclassified = total - classified

status_df = pd.DataFrame({
    'Status'     : ['Classified', 'Unclassified', 'Total'],
    'Row Count'  : [classified, unclassified, total],
    'Percentage' : [f'{classified/total*100:.1f}%', f'{unclassified/total*100:.1f}%', '100%']
})

print("AUTO_FAILURE_REASON — Classification status \n")
print(status_df.to_string(index=False))

AUTO_FAILURE_REASON — Classification status 

      Status  Row Count Percentage
  Classified       6104      44.8%
Unclassified       7520      55.2%
       Total      13624       100%


In [26]:
# Deriving the 24 target rows — step by step

step1 = his_exec.copy()
step2 = step1[step1['AUTO_FAILURE_REASON'].fillna('').str.strip() == '']
step3 = step2[step2['INTRIM_STATUS'] == 'PASSED']
step4 = step2[step2['INTRIM_STATUS'] != 'PASSED']

print(f"Step 1 — Start with all rows in HIS_EXEC_REPORT                                          : {len(step1)}")
print(f"Step 2 — Remove rows already classified (AUTO_FAILURE_REASON filled by engineers)         : {len(step1) - len(step2)} removed → {len(step2)} remaining")
print(f"Step 3 — You're left with unlabeled rows                                                  : {len(step2)}")
print(f"Step 4 — Remove PASSED rows (INTRIM_STATUS = PASSED — test passed, nothing to classify)   : {len(step3)} removed → {len(step4)} remaining")
print(f"Step 5 — What remains = {len(step4)} rows (FAILED, BLOCKED, OOS, SCHEDULED) with no classification yet")
print()
print("These are the rows our system needs to classify.")

Step 1 — Start with all rows in HIS_EXEC_REPORT                                          : 13624
Step 2 — Remove rows already classified (AUTO_FAILURE_REASON filled by engineers)         : 6104 removed → 7520 remaining
Step 3 — You're left with unlabeled rows                                                  : 7520
Step 4 — Remove PASSED rows (INTRIM_STATUS = PASSED — test passed, nothing to classify)   : 7496 removed → 24 remaining
Step 5 — What remains = 24 rows (FAILED, BLOCKED, OOS, SCHEDULED) with no classification yet

These are the rows our system needs to classify.


In [ ]:
both = (his_exec['FAILURE_REMARKS'].fillna('').str.strip() != '') & (his_exec['AUTO_FAILURE_REASON'].fillna('').str.strip() != '')
print(f"Rows with both FAILURE_REMARKS and AUTO_FAILURE_REASON: {both.sum()}")